In [ ]:
%load_ext autoreload

from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv("../.env")

In [ ]:
%autoreload 2

import json
import random
from pathlib import Path

import folium
import geopandas as gpd
import matplotlib.colors as colors
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
ca = gpd.read_file("/Volumes/x10pro/estuary/geos/ca_geo.geojson")
pmep_points = gpd.read_file("/Volumes/x10pro/estuary/geos/PMEP_Estuary_Points_V1_3.gdb/").to_crs(
    ca.crs
)

# Goleta Slough too close to San Pedro Creek, we see both!
pmep_points = pmep_points[pmep_points.PMEP_EstuaryID != 4003].copy()


def read_grid(p):
    gdf = gpd.read_file(p).to_crs(ca.crs)
    gdf["Site code"] = int(p.stem)
    return gdf


grids = gpd.GeoDataFrame(
    pd.concat(
        [read_grid(p) for p in Path("/Volumes/x10pro/estuary/ca_grids/").glob("*.geojson")],
        ignore_index=True,
    ),
    geometry="geometry",
).to_crs(ca.crs)
grids["skipped"] = False

skipped_grids = gpd.GeoDataFrame(
    pd.concat(
        [read_grid(p) for p in Path("/Volumes/x10pro/estuary/skipped_grids/").glob("*.geojson")],
        ignore_index=True,
    ),
    geometry="geometry",
).to_crs(ca.crs)
skipped_grids["skipped"] = True

grids = gpd.GeoDataFrame(
    pd.concat([grids, skipped_grids], ignore_index=True), geometry="geometry"
).to_crs(ca.crs)

pmep_points = pmep_points[pmep_points.intersects(ca.iloc[0].geometry)].copy()

In [ ]:
grids.head(5)

In [ ]:
# Example: pick something appropriate for your region, like a UTM zone
gdf_pts = pmep_points.to_crs("EPSG:32610")  # just an example
gdf_grid = grids.to_crs(gdf_pts.crs)

gdf_pts["name_norm"] = gdf_pts["Estuary_Name"].str.lower().str.strip()
gdf_grid["name_norm"] = gdf_grid["Site name"].str.lower().str.strip()

name_join = gdf_pts.merge(
    gdf_grid,  # include other grid columns if needed
    on="name_norm",
    how="left",
    suffixes=("", "_grid"),
    right_index=False,
)

matched_by_name = name_join[~name_join["geometry_grid"].isna()].copy()
unmatched_pts = (
    name_join[name_join["geometry_grid"].isna()]
    .copy()
    .drop(columns=["Site code", "Site name", "skipped", "geometry_grid"])
)

# Left join: keep all points, attach polygon attrs where they intersect
pts_with_grid = gpd.sjoin(
    unmatched_pts,
    gdf_grid.drop(columns=["name_norm"]),
    how="left",
    predicate="intersects",
).drop(columns=["index_right"])

len(matched_by_name), (~pts_with_grid["Site code"].isna()).sum(), len(gdf_grid)

In [ ]:
no_grid_pts = (
    pts_with_grid[pts_with_grid["Site code"].isna()]
    .copy()
    .drop(columns=["Site code", "Site name", "skipped"])
)

N = 1500  # meters

grid_buffer = gdf_grid.copy()
grid_buffer["geometry"] = grid_buffer.buffer(N)

near_pts = gpd.sjoin(
    no_grid_pts.drop(columns=["name_norm"]),
    grid_buffer.drop(columns=["name_norm"]),
    how="inner",
    predicate="intersects",
).drop(columns=["index_right"])

len(near_pts)

In [ ]:
m = gdf_grid.explore(style_kwds={"fillOpacity": 0, "color": "black"})
near_pts.explore(m=m, marker_kwds={"radius": 6})

In [ ]:
remove = [4048, 4041, 4011, 4008, 4006, 3096, 3073, 3074, 3071, 3067, 3043, 3027, 3025]

# remove = [4048, 4011, 4006, 4008, 3073, 3074, 3071, 3067, 3025, 3027, 3043, 3096, 4041, 4014, 3096, 3089, 3076, 3073, 3065, 3010]

In [ ]:
near_pts = near_pts[~near_pts.PMEP_EstuaryID.isin(remove)].copy()
intersect_points = pts_with_grid[~pts_with_grid["Site code"].isna()].copy()
matches = gpd.GeoDataFrame(
    pd.concat(
        [
            near_pts,
            intersect_points.drop(columns=["name_norm"]),
            matched_by_name.drop(columns=["name_norm", "geometry_grid"]),
        ]
    ),
    geometry="geometry",
    crs=intersect_points.crs,
)
matches["skipped"] = matches.skipped.apply(bool)
print(len(matches))
matches.head()

In [ ]:
matched_valid = matches["Site code"].unique().astype(int)
print(len(matched_valid), sum(~gdf_grid.skipped))

gdf_grid[(~gdf_grid["Site code"].isin(matched_valid)) & (gdf_grid.skipped == False)]

In [ ]:
missed_grids = gdf_grid[~gdf_grid["Site code"].isin(matches["Site code"])]

m = gdf_pts[~gdf_pts.PMEP_EstuaryID.isin(matches.PMEP_EstuaryID)].explore(marker_kwds={"radius": 6})
missed_grids[missed_grids.skipped == False].explore(
    m=m, style_kwds={"fillOpacity": 0, "color": "black"}
)

In [ ]:
valid_matches = matches[matches.skipped == False]
valid_matches.Estuary_Hectares.hist()

In [ ]:
valid_matches[valid_matches.Estuary_Hectares < 5]

In [ ]:
unmatched_pts = gdf_pts[~gdf_pts.PMEP_EstuaryID.isin(matches.PMEP_EstuaryID)]
unmatched_pts[
    (unmatched_pts.Estuary_Hectares < 1000) & (unmatched_pts.Estuary_Hectares > 5)
].Estuary_Hectares.hist()

In [ ]:
potentials = unmatched_pts[
    (unmatched_pts.Estuary_Hectares > 5) & (unmatched_pts.CMECS_Class == "Lagoonal Estuary")
]

print(len(potentials))

potentials.explore(marker_kwds={"radius": 6})

In [ ]:
maybe_add = [2097, 2096, 2103, 2105, 3008, 3009, 3028, 3027, 3057, 3073, 3099, 4054]

sorted(potentials[potentials.PMEP_EstuaryID.isin(maybe_add)].Estuary_Name.tolist())

In [ ]:
grids_data = (
    gpd.read_file("/Volumes/x10pro/estuary/geos/ca_data.geojson")
    .drop(columns=["Nation", "Region", "Studied (1) / Not Studied (0)", "Longitude", "Latitude"])
    .set_index("Site code")
    .to_crs(ca.crs)
    .rename(columns={"geometry": "point_geom"})
    .join(grids.set_index("Site code").drop(columns=["Site name"]), how="inner")
)
grids_data.head()

In [ ]:
matches["Site code"] = matches["Site code"].astype(int)
pmep_merged = grids_data.join(
    matches.set_index("Site code")
    .to_crs(ca.crs)
    .rename(columns={"geometry": "pmep_pt_geom"})
    .drop(columns=["Site name", "skipped"]),
    how="left",
)
pmep_merged["potential_add"] = False
pmep_merged.PMEP_EstuaryID.isna().sum()

In [ ]:
pmep_merged.head(1)

In [ ]:
to_add = potentials[potentials.PMEP_EstuaryID.isin(maybe_add)].copy().to_crs(pmep_merged.crs)
to_add["potential_add"] = True
to_add["Site code"] = to_add.PMEP_EstuaryID + 10000
to_add["Site name"] = to_add.Estuary_Name
to_add["point_geom"] = to_add.geometry
to_add["pmep_pt_geom"] = to_add.geometry
to_add["skipped"] = False
to_add = to_add.set_index("Site code").drop(columns=["name_norm"])

pmep_merged_with_extra = pd.concat([pmep_merged, to_add])

pmep_merged_with_extra.head(3)

In [ ]:
df = pd.read_csv("/Volumes/x10pro/estuary/ca_all/empa/grab_events.csv")
df = df[(df.latitude > -87) & (df.longitude < 100)].drop_duplicates(["siteid"])
df = df[["siteid", "estuaryname", "latitude", "longitude"]]

empa_sites = (
    gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df["longitude"], df["latitude"]),
        crs="EPSG:4326",
    )
    .drop(columns=["latitude", "longitude"])
    .sort_values("siteid")
    .reset_index(drop=True)
)
empa_sites["name_norm"] = empa_sites["estuaryname"].str.lower().str.strip()
empa_sites.head()
# gdf.explore(marker_kwds={"radius": 6, 'color': 'blue'})

In [ ]:
# empa_sites = gpd.read_file("/Volumes/x10pro/estuary/geos/empa_sites.geojson").to_crs(ca.crs)
# empa_sites["name_norm"] = empa_sites["estuaryname"].str.lower().str.strip()
# empa_sites.head()

In [ ]:
aaa = pmep_merged_with_extra.reset_index()
aaa["name_norm"] = aaa["Site name"].str.lower().str.strip()

name_join = aaa.merge(
    empa_sites.drop_duplicates(["siteid", "estuaryname"])[["name_norm", "siteid"]],
    on="name_norm",
    how="left",
    right_index=False,
)
name_match = name_join[~name_join.siteid.isna()].copy()
name_miss = name_join[name_join.siteid.isna()].drop(columns=["siteid"])
name_miss["name_norm"] = name_miss["Estuary_Name"].str.lower().str.strip()

name_join2 = name_miss.merge(
    empa_sites.drop_duplicates(["siteid", "estuaryname"])[["name_norm", "siteid"]],
    on="name_norm",
    how="left",
    right_index=False,
)

pmep_empa_name_merged = (
    pd.concat([name_match, name_join2], ignore_index=True)
    .set_index("Site code")
    .drop(columns=["name_norm"])
)

pmep_empa_name_merged.head()

In [ ]:
with open("/Volumes/x10pro/estuary/geos/ca_empa_matching_sites.json") as f:
    prev_matches = json.load(f)

for k, v in prev_matches.items():
    pmep_empa_name_merged.loc[int(k), "siteid"] = v

In [ ]:
sites_search = pmep_empa_name_merged[pmep_empa_name_merged.siteid.isna()].copy()
empa_search = empa_sites[~empa_sites.siteid.isin(pmep_empa_name_merged.siteid)].copy()

sites_search["geometry"] = sites_search.point_geom


m = sites_search.explore(marker_kwds={"radius": 6, "color": "blue"})
empa_search.explore(marker_kwds={"radius": 6, "color": "red"}, m=m)

In [ ]:
pmep_empa_name_merged[
    (~pmep_empa_name_merged.siteid.isna()) & (pmep_empa_name_merged.potential_add)
]

In [ ]:
pmep_empa_name_merged[~pmep_empa_name_merged.siteid.isna()]

In [ ]:
nearests = (
    gpd.sjoin_nearest(
        sites_search.to_crs("EPSG:32610"),
        empa_search[["geometry", "siteid", "estuaryname"]].to_crs("EPSG:32610"),
        how="inner",
        max_distance=2000,
        distance_col="dist",
    )
    # .sort_values(by=["dist", "siteid"])
    # .drop_duplicates(["siteid", "Site code"])
)
# Carpinteria Creek & Carpinteria Estuary are different!
nearests

In [ ]:
import polars as pl

empa = pl.read_csv("/Volumes/x10pro/estuary/ca_all/empa/logger-raw-depth-correction-publish.csv")
empa_unique = empa.unique(subset=["siteid"])
empa_a = empa_unique.to_pandas()[["estuaryname", "siteid"]]

empa = pl.read_csv("/Volumes/x10pro/estuary/ca_all/empa/logger-raw-publish.csv")
empa_unique = empa.unique(subset=["siteid"])
empa_b = empa_unique.to_pandas()[["estuaryname", "siteid"]]

empa = pl.read_csv("/Volumes/x10pro/estuary/ca_all/emailed_water_data.csv")
empa_unique = empa.unique(subset=["siteid"])
empa_c = empa_unique.to_pandas()[["estuaryname", "siteid"]]

empa_sites_cat = (
    pd.concat([empa_a, empa_b, empa_c], ignore_index=True)
    .sort_values(["estuaryname"])
    .drop_duplicates("estuaryname")
    .reset_index(drop=True)
)

empa_sites_cat

In [ ]:
empa_sites_cat["nocreek"] = empa_sites_cat.estuaryname.apply(lambda n: n.split(" Creek")[0])
pmep_empa_name_merged["nocreek"] = (
    pmep_empa_name_merged["Estuary_Name"]
    .fillna(pmep_empa_name_merged["Site name"])
    .fillna("")
    .apply(lambda n: n.split(" Creek")[0])
)
names = (
    empa_sites_cat[~empa_sites_cat.siteid.isin(pmep_empa_name_merged.siteid)]
    .nocreek.unique()
    .tolist()
)

new_matches = pmep_empa_name_merged[
    (pmep_empa_name_merged.nocreek.isin(names)) & (pmep_empa_name_merged.siteid.isna())
]
misses = [n for n in names if n not in new_matches.nocreek.tolist()]

nocreek_to_siteid = (
    empa_sites_cat.dropna(subset=["siteid"])
    .drop_duplicates("nocreek")
    .set_index("nocreek")["siteid"]
)
mask = pmep_empa_name_merged["siteid"].isna()

pmep_empa_name_merged.loc[mask, "siteid"] = pmep_empa_name_merged.loc[mask, "nocreek"].map(
    nocreek_to_siteid
)

pmep_empa_name_merged = pmep_empa_name_merged.drop(columns=["nocreek"])

In [ ]:
# List of sites with empa matches and water data
matched_water_sites = pmep_empa_name_merged[
    pmep_empa_name_merged.siteid.isin(empa_sites_cat.siteid)
].Estuary_Name.tolist()
matched_water_sites.sort()

for p in matched_water_sites:
    print(p)

In [ ]:
# list of sites that have empa matches but dont have water data

for p in pmep_empa_name_merged[
    (~pmep_empa_name_merged.siteid.isin(empa_sites_cat.siteid))
    & (~pmep_empa_name_merged.siteid.isna())
]["Site name"].tolist():
    print(p)

# Missing EMPA Data

Los Penasquitos Lagoon
Zuma Lagoon
Russian River
San Dieguito Lagoon
San Elijo Lagoon
Deveereux Slough
Mugu Lagoon
Tijuana River Estuary

In [ ]:
pmep_empa_name_merged["point_geom_lat"] = pmep_empa_name_merged.point_geom.y
pmep_empa_name_merged["point_geom_long"] = pmep_empa_name_merged.point_geom.x
pmep_empa_name_merged["pmep_pt_geom_lat"] = pmep_empa_name_merged.pmep_pt_geom.y
pmep_empa_name_merged["pmep_pt_geom_long"] = pmep_empa_name_merged.pmep_pt_geom.x

pmep_empa_name_merged.drop(columns=["point_geom", "pmep_pt_geom"]).to_file(
    "/Volumes/x10pro/estuary/geos/ca_data_w_empa_pmep.geojson"
)

In [ ]:
site_words = [
    "Earl",
    "Crescent",
    "Humboldt",
    "Cleone",
    "Noyo",
    "Russian river",
    "Salmon",
    "Bodega",
    "Tomales",
    "Petaluma",
    "Novato",
    "Coyote",
    "Gregorio",
    "Pescadero",
    "Younger",
    "Aptos",
    "Elkhorn",
    "Carmel",
    "Morro",
    "Grande",
    "Devereux",
    "Ventura",
    "Clara",
    "Newport",
    "Penasquitos",
    "Diego",
    "Tijuana",
]


def name_matches(n):
    if isinstance(n, float):
        return False
    return any(c.lower() in n.lower() for c in site_words)


a = pmep_empa_name_merged[pmep_empa_name_merged.Estuary_Name.apply(name_matches)]
# a = a[a.siteid.isna()]
b = a[["Estuary_Name"]].sort_values(by="Estuary_Name")
for idx, p in b.iterrows():
    print(idx, p.Estuary_Name)

# UC Davis Water
https://coastalocean.ucdavis.edu/ocean-observing/water-level
- 34 Arroyo Grande Creek Lagoon
- 48 Carmel River
- 25 Devereux Lagoon
- 98 Lake Earl
- 11 Los Penasquitos Lagoon
- 2138 Pescadero Creek
- 72 Russian River
- 70 Salmon Creek
- 13057 San Gregorio Creek
- 20 Santa Clara River
- 2163 Tijuana River
- 21 Ventura River
- 53 Younger Lagoon

In [ ]:
lit = gpd.read_file(
    "/Volumes/x10pro/estuary/geos/Littoral_Cells_2005/Littoral_Cells_20120419.shp"
).to_crs(4326)
# lit.head()

In [ ]:
small = gpd.GeoDataFrame(
    pd.concat(
        [
            gpd.read_file(p).to_crs(4326)
            for p in Path("/Volumes/x10pro/estuary/ca_grids/").glob("*.geojson")
        ]
    ),
    geometry="geometry",
)


# Use centroids for small grids if they are polygons
small_pts = small.copy()
if not small_pts.geometry.iloc[0].geom_type == "Point":
    small_pts["geometry"] = small_pts.geometry.centroid

# Spatial join: nearest lit polygon
small_assigned = gpd.sjoin_nearest(
    small_pts,
    lit[["CELL_NAME", "geometry"]],  # keep only what we need
    how="left",
    distance_col="dist_to_lit",
)

In [ ]:
# Initialize map centered on all data
bounds = lit.total_bounds  # [minx, miny, maxx, maxy]
m = folium.Map(
    location=[(bounds[1] + bounds[3]) / 2, (bounds[0] + bounds[2]) / 2],
    zoom_start=5,
    tiles="CartoDB positron",
    width=700,
    height=500,
)

# Map each CELL_NAME to a color
unique = lit["CELL_NAME"].unique().tolist()
random.shuffle(unique)
cmap = plt.get_cmap("tab20", len(unique))
norm = colors.Normalize(vmin=0, vmax=len(unique))

name_to_color = {name: colors.to_hex(cmap(i)) for i, name in enumerate(unique)}

for _, row in lit.iterrows():
    color = name_to_color[row["CELL_NAME"]]
    folium.GeoJson(
        row.geometry,
        name=row["CELL_NAME"],
        popup=folium.Popup(row["CELL_NAME"]),
        style_function=lambda x, color=color: {
            "fillColor": color,
            "color": None,
            "weight": 1,
            "fillOpacity": 0.4,
        },
    ).add_to(m)

# --- Plot small grids as points, colored by assigned lit cell ---
for _, row in small_assigned.iterrows():
    cell_name = row["CELL_NAME"]  # from lit via sjoin_nearest
    grid_name = row["Site name"]  # from lit via sjoin_nearest
    color = name_to_color.get(cell_name, "#000000")
    geom = row.geometry  # centroid point

    folium.CircleMarker(
        location=[geom.y, geom.x],
        radius=4,
        popup=folium.Popup(grid_name),
        fill=True,
        fill_color=color,
        color="black",  # ← visible border
        weight=1.5,  # ← stronger outline
        fill_opacity=0.9,
    ).add_to(m)

m